In [ ]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [33]:
import json
import pandas as pd
import os

print("✅ Core imports loaded")

✅ Core imports loaded


In [ ]:
!pip install -q fastapi uvicorn pandas numpy requests python-dotenv pydantic

In [ ]:
import os

folders = [
    "recoverai",
    "recoverai/data",
    "recoverai/agent",
    "recoverai/tools",
    "recoverai/backend",
    "recoverai/frontend",
    "recoverai/database"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created!")

Project folders created!


In [ ]:
!find recoverai -maxdepth 3 -type d

recoverai
recoverai/data
recoverai/agent
recoverai/database
recoverai/backend
recoverai/tools
recoverai/frontend


In [5]:
import pandas as pd

transactions = pd.DataFrame([
    {
        "transaction_id": "TXN001",
        "customer_id": "CUS001",
        "customer_name": "Rahul Sharma",
        "amount": 2499,
        "status": "failed",
        "failure_reason": "bank_declined",
        "attempt_count": 1
    },
    {
        "transaction_id": "TXN002",
        "customer_id": "CUS002",
        "customer_name": "Aisha Khan",
        "amount": 999,
        "status": "failed",
        "failure_reason": "expired_card",
        "attempt_count": 1
    },
    {
        "transaction_id": "TXN003",
        "customer_id": "CUS003",
        "customer_name": "Vikram Singh",
        "amount": 5999,
        "status": "failed",
        "failure_reason": "multiple_failures",
        "attempt_count": 3
    },
    {
        "transaction_id": "TXN004",
        "customer_id": "CUS004",
        "customer_name": "Sara Patel",
        "amount": 1499,
        "status": "abandoned",
        "failure_reason": "checkout_abandoned",
        "attempt_count": 0
    },
    {
        "transaction_id": "TXN005",
        "customer_id": "CUS005",
        "customer_name": "Arjun Mehta",
        "amount": 3999,
        "status": "failed",
        "failure_reason": "authentication_timeout",
        "attempt_count": 1
    }
])

transactions

,transaction_id,customer_id,customer_name,amount,status,failure_reason,attempt_count
0,TXN001,CUS001,Rahul Sharma,2499,failed,bank_declined,1
1,TXN002,CUS002,Aisha Khan,999,failed,expired_card,1
2,TXN003,CUS003,Vikram Singh,5999,failed,multiple_failures,3
3,TXN004,CUS004,Sara Patel,1499,abandoned,checkout_abandoned,0
4,TXN005,CUS005,Arjun Mehta,3999,failed,authentication_timeout,1


In [6]:
customers = pd.DataFrame([
    {
        "customer_id": "CUS001",
        "customer_name": "Rahul Sharma",
        "successful_payments": 7,
        "failed_payments": 1,
        "lifetime_value": 18000
    },
    {
        "customer_id": "CUS002",
        "customer_name": "Aisha Khan",
        "successful_payments": 4,
        "failed_payments": 1,
        "lifetime_value": 7500
    },
    {
        "customer_id": "CUS003",
        "customer_name": "Vikram Singh",
        "successful_payments": 12,
        "failed_payments": 3,
        "lifetime_value": 42000
    },
    {
        "customer_id": "CUS004",
        "customer_name": "Sara Patel",
        "successful_payments": 5,
        "failed_payments": 0,
        "lifetime_value": 12500
    },
    {
        "customer_id": "CUS005",
        "customer_name": "Arjun Mehta",
        "successful_payments": 9,
        "failed_payments": 1,
        "lifetime_value": 29000
    }
])

customers

,customer_id,customer_name,successful_payments,failed_payments,lifetime_value
0,CUS001,Rahul Sharma,7,1,18000
1,CUS002,Aisha Khan,4,1,7500
2,CUS003,Vikram Singh,12,3,42000
3,CUS004,Sara Patel,5,0,12500
4,CUS005,Arjun Mehta,9,1,29000


In [7]:
transactions.to_csv(
    "recoverai/data/transactions.csv",
    index=False
)

customers.to_csv(
    "recoverai/data/customers.csv",
    index=False
)

print("Data saved successfully!")

Data saved successfully!


In [ ]:
def analyze_payment(transaction, customer):
    """
    Analyze a failed/abandoned payment
    and determine an appropriate recovery strategy.
    """

    reason = transaction["failure_reason"]
    amount = transaction["amount"]
    attempts = transaction["attempt_count"]

    if reason == "bank_declined" and attempts <= 1:
        strategy = "retry_payment"
        priority = "medium"

    elif reason == "expired_card":
        strategy = "request_payment_method_update"
        priority = "medium"

    elif reason == "multiple_failures":
        strategy = "escalate_to_human"
        priority = "high"

    elif reason == "checkout_abandoned":
        strategy = "send_recovery_message"
        priority = "medium"

    elif reason == "authentication_timeout":
        strategy = "retry_payment"
        priority = "medium"

    else:
        strategy = "human_review"
        priority = "high"

    return {
        "customer": customer["customer_name"],
        "amount": amount,
        "failure_reason": reason,
        "recommended_strategy": strategy,
        "priority": priority
    }

In [ ]:
transaction = transactions.iloc[0]
customer = customers[
    customers["customer_id"] == transaction["customer_id"]
].iloc[0]

result = analyze_payment(transaction, customer)

result

{'customer': 'Rahul Sharma',
 'amount': np.int64(2499),
 'failure_reason': 'bank_declined',
 'recommended_strategy': 'retry_payment',
 'priority': 'medium'}

In [ ]:
!pip install -q openai

In [ ]:
import openai

print("OpenAI package installed successfully!")

OpenAI package installed successfully!


In [ ]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key was not found.")

API key loaded successfully!


In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

print("AI client initialized successfully!")

AI client initialized successfully!


In [ ]:
SYSTEM_PROMPT = """
You are RecoverAI, an AI revenue recovery agent for online merchants.

Your job is to analyze failed or abandoned payments and recommend
the safest and most effective recovery strategy.

You must consider:

1. Payment failure reason
2. Payment amount
3. Number of payment attempts
4. Customer's successful payment history
5. Customer's failed payment history
6. Customer lifetime value

Available recovery strategies:

- retry_payment
- request_payment_method_update
- send_recovery_message
- escalate_to_human
- human_review

Safety rules:

- Never recommend unlimited payment retries.
- Repeated payment failures should be escalated.
- High-value or unusual cases should receive human review.
- Explain why the selected strategy is appropriate.

Return the result as JSON with these fields:

recovery_probability
strategy
priority
reason
requires_human_approval
"""

In [ ]:
import json

def ai_analyze_payment(transaction, customer):

    payment_data = {
        "transaction_id": str(transaction["transaction_id"]),
        "customer_id": str(transaction["customer_id"]),
        "customer_name": str(customer["customer_name"]),
        "amount": int(transaction["amount"]),
        "status": str(transaction["status"]),
        "failure_reason": str(transaction["failure_reason"]),
        "attempt_count": int(transaction["attempt_count"]),
        "successful_payments": int(customer["successful_payments"]),
        "failed_payments": int(customer["failed_payments"]),
        "lifetime_value": int(customer["lifetime_value"])
    }

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(payment_data)
            }
        ]
    )

    return json.loads(
        response.choices[0].message.content
    )

In [ ]:
def recoverai_decision(transaction, customer):
    """
    RecoverAI decision engine.

    Analyzes a failed/abandoned payment and determines
    the safest recovery strategy.
    """

    reason = str(transaction["failure_reason"])
    amount = int(transaction["amount"])
    attempts = int(transaction["attempt_count"])

    successful_payments = int(customer["successful_payments"])
    failed_payments = int(customer["failed_payments"])
    lifetime_value = int(customer["lifetime_value"])

    # Calculate a simple customer reliability score
    total_payments = successful_payments + failed_payments

    if total_payments > 0:
        success_rate = successful_payments / total_payments
    else:
        success_rate = 0

    # -----------------------------
    # Recovery decision logic
    # -----------------------------

    if attempts >= 3:
        strategy = "escalate_to_human"
        priority = "high"
        reason_text = (
            "Payment has already been attempted multiple times. "
            "Further automatic retries should be avoided."
        )

    elif reason == "expired_card":
        strategy = "request_payment_method_update"
        priority = "medium"
        reason_text = (
            "The customer's payment method appears to be expired. "
            "The customer should update their payment method."
        )

    elif reason == "checkout_abandoned":
        strategy = "send_recovery_message"
        priority = "medium"
        reason_text = (
            "The customer abandoned checkout. "
            "A recovery message can encourage them to complete the payment."
        )

    elif reason == "bank_declined" and attempts <= 1:
        strategy = "retry_payment"
        priority = "medium"
        reason_text = (
            "The payment was declined once and has not exceeded "
            "the retry limit, so a controlled retry is appropriate."
        )

    elif reason == "authentication_timeout":
        strategy = "retry_payment"
        priority = "medium"
        reason_text = (
            "Authentication timed out. A controlled retry may allow "
            "the customer to complete authentication."
        )

    elif reason == "multiple_failures":
        strategy = "escalate_to_human"
        priority = "high"
        reason_text = (
            "Multiple payment failures indicate that automatic recovery "
            "may not be appropriate."
        )

    else:
        strategy = "human_review"
        priority = "high"
        reason_text = (
            "The failure reason is not recognized by the automated "
            "recovery rules, so human review is safer."
        )

    # High-value transaction safety rule
    if amount >= 20000:
        priority = "high"

        if strategy == "retry_payment":
            strategy = "human_review"
            reason_text += (
                " The transaction value is high, so human approval "
                "is required before retrying."
            )

    return {
        "customer": str(customer["customer_name"]),
        "customer_id": str(customer["customer_id"]),
        "amount": amount,
        "failure_reason": reason,
        "attempt_count": attempts,
        "success_rate": round(success_rate, 2),
        "lifetime_value": lifetime_value,
        "recommended_strategy": strategy,
        "priority": priority,
        "reason": reason_text,
        "requires_human_approval": (
            strategy in ["human_review", "escalate_to_human"]
        )
    }

In [ ]:
transaction = transactions.iloc[0]

customer = customers[
    customers["customer_id"] == transaction["customer_id"]
].iloc[0]

decision = recoverai_decision(
    transaction,
    customer
)

print(json.dumps(decision, indent=2))

{
  "customer": "Rahul Sharma",
  "customer_id": "CUS001",
  "amount": 2499,
  "failure_reason": "bank_declined",
  "attempt_count": 1,
  "success_rate": 0.88,
  "lifetime_value": 18000,
  "recommended_strategy": "retry_payment",
  "priority": "medium",
  "reason": "The payment was declined once and has not exceeded the retry limit, so a controlled retry is appropriate.",
  "requires_human_approval": false
}


In [20]:
def get_customer_history(customer_id):
    """
    Tool: Retrieve customer payment history.
    """

    customer_rows = customers[
        customers["customer_id"] == customer_id
    ]

    if len(customer_rows) == 0:
        return {
            "success": False,
            "message": "Customer not found"
        }

    customer = customer_rows.iloc[0]

    return {
        "success": True,
        "customer_id": str(customer["customer_id"]),
        "customer_name": str(customer["customer_name"]),
        "successful_payments": int(customer["successful_payments"]),
        "failed_payments": int(customer["failed_payments"]),
        "lifetime_value": int(customer["lifetime_value"])
    }

In [ ]:
history = get_customer_history(
    str(transaction["customer_id"])
)

print(json.dumps(history, indent=2))

{
  "success": true,
  "customer_id": "CUS001",
  "customer_name": "Rahul Sharma",
  "successful_payments": 7,
  "failed_payments": 1,
  "lifetime_value": 18000
}


In [19]:
def get_payment_details(transaction_id):
    """
    Tool: Retrieve payment details.
    """

    rows = transactions[
        transactions["transaction_id"].astype(str) == str(transaction_id)
    ]

    if len(rows) == 0:
        return {
            "success": False,
            "message": "Transaction not found"
        }

    transaction = rows.iloc[0]

    return {
        "success": True,
        "transaction_id": str(transaction["transaction_id"]),
        "customer_id": str(transaction["customer_id"]),
        "amount": int(transaction["amount"]),
        "status": str(transaction["status"]),
        "failure_reason": str(transaction["failure_reason"]),
        "attempt_count": int(transaction["attempt_count"])
    }

In [ ]:
payment = get_payment_details(
    str(transaction["transaction_id"])
)

print(json.dumps(payment, indent=2))

{
  "success": true,
  "transaction_id": "TXN001",
  "customer_id": "CUS001",
  "amount": 2499,
  "status": "failed",
  "failure_reason": "bank_declined",
  "attempt_count": 1
}


In [22]:
def retry_payment(transaction_id):
    """
    Simulated payment retry tool.

    This does NOT charge a real customer.
    """

    rows = transactions[
        transactions["transaction_id"].astype(str) == str(transaction_id)
    ]

    if len(rows) == 0:
        return {
            "success": False,
            "message": "Transaction not found"
        }

    transaction_index = rows.index[0]

    current_attempts = int(
        transactions.loc[transaction_index, "attempt_count"]
    )

    if current_attempts >= 3:
        return {
            "success": False,
            "message": "Retry limit reached. Human review required."
        }

    # Simulate another attempt
    transactions.loc[
        transaction_index,
        "attempt_count"
    ] = current_attempts + 1

    return {
        "success": True,
        "action": "retry_payment",
        "transaction_id": str(transaction_id),
        "attempt_number": current_attempts + 1,
        "message": "Payment retry simulated successfully."
    }

In [23]:
def send_recovery_message(customer_id, amount):
    """
    Simulated customer recovery message tool.
    """

    history = get_customer_history(customer_id)

    if not history["success"]:
        return {
            "success": False,
            "message": "Customer not found"
        }

    customer_name = history["customer_name"]

    message = (
        f"Hi {customer_name}, we noticed that your payment of "
        f"₹{amount} was not completed. "
        f"Please return to checkout to complete your payment."
    )

    return {
        "success": True,
        "action": "send_recovery_message",
        "customer_id": customer_id,
        "message": message
    }

In [24]:
def escalate_to_human(transaction_id, reason):
    """
    Simulated human escalation tool.
    """

    return {
        "success": True,
        "action": "escalate_to_human",
        "transaction_id": str(transaction_id),
        "reason": reason,
        "status": "pending_human_review"
    }

In [ ]:
def recoverai_agent(transaction_id):
    """
    RecoverAI Agent

    Workflow:
    1. Get payment
    2. Get customer
    3. Analyze situation
    4. Select strategy
    5. Execute appropriate tool
    6. Return complete agent result
    """

    # --------------------------------
    # STEP 1: Get payment information
    # --------------------------------

    payment = get_payment_details(transaction_id)

    if not payment["success"]:
        return payment

    # --------------------------------
    # STEP 2: Get customer information
    # --------------------------------

    customer = get_customer_history(
        payment["customer_id"]
    )

    if not customer["success"]:
        return customer

    # --------------------------------
    # STEP 3: Analyze payment
    # --------------------------------

    decision = recoverai_decision(
        transactions[
            transactions["transaction_id"].astype(str)
            == str(transaction_id)
        ].iloc[0],
        customers[
            customers["customer_id"].astype(str)
            == str(payment["customer_id"])
        ].iloc[0]
    )

    strategy = decision["recommended_strategy"]

    # --------------------------------
    # STEP 4: Execute selected tool
    # --------------------------------

    if strategy == "retry_payment":

        action_result = retry_payment(
            transaction_id
        )

    elif strategy == "send_recovery_message":

        action_result = send_recovery_message(
            payment["customer_id"],
            payment["amount"]
        )

    elif strategy == "escalate_to_human":

        action_result = escalate_to_human(
            transaction_id,
            decision["reason"]
        )

    elif strategy == "request_payment_method_update":

        action_result = {
            "success": True,
            "action": "request_payment_method_update",
            "customer_id": payment["customer_id"],
            "message": "Customer should update their payment method."
        }

    else:

        action_result = {
            "success": True,
            "action": "human_review",
            "message": "Payment requires human review."
        }

    # --------------------------------
    # STEP 5: Return complete result
    # --------------------------------

    return {
        "agent": "RecoverAI",
        "transaction": payment,
        "customer": customer,
        "decision": decision,
        "action": action_result
    }

In [ ]:
agent_result = recoverai_agent(
    str(transaction["transaction_id"])
)

print(json.dumps(agent_result, indent=2))

{
  "agent": "RecoverAI",
  "transaction": {
    "success": true,
    "transaction_id": "TXN001",
    "customer_id": "CUS001",
    "amount": 2499,
    "status": "failed",
    "failure_reason": "bank_declined",
    "attempt_count": 1
  },
  "customer": {
    "success": true,
    "customer_id": "CUS001",
    "customer_name": "Rahul Sharma",
    "successful_payments": 7,
    "failed_payments": 1,
    "lifetime_value": 18000
  },
  "decision": {
    "customer": "Rahul Sharma",
    "customer_id": "CUS001",
    "amount": 2499,
    "failure_reason": "bank_declined",
    "attempt_count": 1,
    "success_rate": 0.88,
    "lifetime_value": 18000,
    "recommended_strategy": "retry_payment",
    "priority": "medium",
    "reason": "The payment was declined once and has not exceeded the retry limit, so a controlled retry is appropriate.",
    "requires_human_approval": false
  },
  "action": {
    "success": true,
    "action": "retry_payment",
    "transaction_id": "TXN001",
    "attempt_number":

In [ ]:
import pandas as pd
from datetime import datetime

recovery_logs = pd.DataFrame(columns=[
    "timestamp",
    "transaction_id",
    "customer_id",
    "customer_name",
    "amount",
    "failure_reason",
    "attempt_count",
    "recommended_strategy",
    "priority",
    "reason",
    "requires_human_approval",
    "action",
    "action_success",
    "action_message",
    "recovered_revenue"
])

print("Recovery log created!")

Recovery log created!


In [ ]:
def record_recovery(agent_result):
    """
    Save the RecoverAI agent decision and action
    into the recovery log.
    """

    transaction = agent_result["transaction"]
    customer = agent_result["customer"]
    decision = agent_result["decision"]
    action = agent_result["action"]

    recovered_revenue = 0

    # In our simulation, a successful retry is considered
    # recovered revenue.
    if (
        action.get("success") is True
        and action.get("action") == "retry_payment"
    ):
        recovered_revenue = transaction["amount"]

    new_record = {
        "timestamp": datetime.now().isoformat(),
        "transaction_id": transaction["transaction_id"],
        "customer_id": customer["customer_id"],
        "customer_name": customer["customer_name"],
        "amount": transaction["amount"],
        "failure_reason": transaction["failure_reason"],
        "attempt_count": transaction["attempt_count"],
        "recommended_strategy": decision["recommended_strategy"],
        "priority": decision["priority"],
        "reason": decision["reason"],
        "requires_human_approval": decision["requires_human_approval"],
        "action": action.get("action", "unknown"),
        "action_success": action.get("success", False),
        "action_message": action.get("message", ""),
        "recovered_revenue": recovered_revenue
    }

    return new_record

In [ ]:
new_record = record_recovery(agent_result)

recovery_logs = pd.concat(
    [
        recovery_logs,
        pd.DataFrame([new_record])
    ],
    ignore_index=True
)

recovery_logs

,timestamp,transaction_id,customer_id,customer_name,amount,failure_reason,attempt_count,recommended_strategy,priority,reason,requires_human_approval,action,action_success,action_message,recovered_revenue
0,2026-08-21T16:30:43.470286,TXN001,CUS001,Rahul Sharma,2499,bank_declined,1,retry_payment,medium,The payment was declined once and has not exce...,False,retry_payment,True,Payment retry simulated successfully.,2499


In [ ]:
import os

os.makedirs("/content/recoverai/database", exist_ok=True)

recovery_logs.to_csv(
    "/content/recoverai/database/recovery_logs.csv",
    index=False
)

print("Recovery database saved successfully!")

Recovery database saved successfully!


In [ ]:
import os

print(
    os.path.exists(
        "/content/recoverai/database/recovery_logs.csv"
    )
)

True


In [35]:
failed_transactions = transactions[
    transactions["status"].isin(["failed", "abandoned"])
].copy()

print(
    "Failed/abandoned transactions:",
    len(failed_transactions)
)

Failed/abandoned transactions: 5


In [36]:
all_agent_results = []

for _, row in failed_transactions.iterrows():

    transaction_id = str(row["transaction_id"])

    try:
        result = recoverai_agent_v2(transaction_id)
        all_agent_results.append(result)

    except Exception as e:
        print(
            f"Error processing {transaction_id}: {e}"
        )

print(
    "Transactions processed:",
    len(all_agent_results)
)

Transactions processed: 5


In [39]:
# ============================================================
# RECOVERAI — RECOVERY LOG RECORD FUNCTION
# ============================================================

def record_recovery(result):
    """
    Convert a RecoverAI agent result into a recovery log record.
    """

    transaction = result.get("transaction", {})
    customer = result.get("customer", {})
    decision = result.get("decision", {})
    action = result.get("action", {})

    # Handle nested payment/customer responses
    transaction_id = transaction.get("transaction_id")
    customer_id = (
        transaction.get("customer_id")
        or customer.get("customer_id")
    )

    amount = transaction.get("amount")
    failure_reason = transaction.get("failure_reason")

    return {
        "transaction_id": transaction_id,
        "customer_id": customer_id,
        "amount": amount,
        "failure_reason": failure_reason,
        "recommended_strategy": decision.get(
            "recommended_strategy"
        ),
        "recovery_score": decision.get(
            "recovery_score"
        ),
        "priority": decision.get(
            "priority"
        ),
        "reason": decision.get(
            "reason"
        ),
        "requires_human_approval": decision.get(
            "requires_human_approval"
        ),
        "action": action.get(
            "action",
            decision.get("recommended_strategy")
        ),
        "action_success": action.get(
            "success"
        )
    }

print("✅ record_recovery loaded successfully")

✅ record_recovery loaded successfully


In [42]:
# ============================================================
# RECOVERAI — INITIALIZE RECOVERY LOGS
# ============================================================

import pandas as pd

recovery_logs = pd.DataFrame()

print("✅ recovery_logs initialized")
print("Current recovery events:", len(recovery_logs))

✅ recovery_logs initialized
Current recovery events: 0


In [43]:
for result in all_agent_results:

    record = record_recovery(result)

    recovery_logs = pd.concat(
        [
            recovery_logs,
            pd.DataFrame([record])
        ],
        ignore_index=True
    )

print(
    "Total recovery events:",
    len(recovery_logs)
)

Total recovery events: 5


In [ ]:
recovery_logs = recovery_logs.drop_duplicates(
    subset=["transaction_id"],
    keep="last"
).reset_index(drop=True)

print("Unique recovery events:", len(recovery_logs))

Unique recovery events: 5


In [ ]:
recovery_logs.to_csv(
    "/content/recoverai/database/recovery_logs.csv",
    index=False
)

print("Recovery database saved successfully!")

Recovery database saved successfully!


In [ ]:
recovery_logs

,timestamp,transaction_id,customer_id,customer_name,amount,failure_reason,attempt_count,recommended_strategy,priority,reason,requires_human_approval,action,action_success,action_message,recovered_revenue
0,2026-08-21T16:36:11.366347,TXN001,CUS001,Rahul Sharma,2499,bank_declined,2,human_review,high,The failure reason is not recognized by the au...,True,human_review,True,Payment requires human review.,0
1,2026-08-21T16:36:11.369506,TXN002,CUS002,Aisha Khan,999,expired_card,1,request_payment_method_update,medium,The customer's payment method appears to be ex...,False,request_payment_method_update,True,Customer should update their payment method.,0
2,2026-08-21T16:36:11.371941,TXN003,CUS003,Vikram Singh,5999,multiple_failures,3,escalate_to_human,high,Payment has already been attempted multiple ti...,True,escalate_to_human,True,,0
3,2026-08-21T16:36:11.374436,TXN004,CUS004,Sara Patel,1499,checkout_abandoned,0,send_recovery_message,medium,The customer abandoned checkout. A recovery me...,False,send_recovery_message,True,"Hi Sara Patel, we noticed that your payment of...",0
4,2026-08-21T16:36:11.376771,TXN005,CUS005,Arjun Mehta,3999,authentication_timeout,1,retry_payment,medium,Authentication timed out. A controlled retry m...,False,retry_payment,True,Payment retry simulated successfully.,3999


In [ ]:
total_failed_revenue = recovery_logs["amount"].sum()

total_recovered_revenue = (
    recovery_logs["recovered_revenue"].sum()
)

successful_actions = (
    recovery_logs["action_success"].sum()
)

total_events = len(recovery_logs)

if total_failed_revenue > 0:
    recovery_rate = (
        total_recovered_revenue /
        total_failed_revenue
    ) * 100
else:
    recovery_rate = 0

print("Total failed revenue: ₹", total_failed_revenue)
print("Recovered revenue: ₹", total_recovered_revenue)
print("Successful actions:", successful_actions)
print("Recovery rate:", round(recovery_rate, 2), "%")

Total failed revenue: ₹ 14995
Recovered revenue: ₹ 3999
Successful actions: 5
Recovery rate: 26.67 %


In [ ]:
strategy_counts = (
    recovery_logs["recommended_strategy"]
    .value_counts()
)

print(strategy_counts)

recommended_strategy
human_review                     1
request_payment_method_update    1
escalate_to_human                1
send_recovery_message            1
retry_payment                    1
Name: count, dtype: int64


In [ ]:
strategy_revenue = (
    recovery_logs
    .groupby("recommended_strategy")["recovered_revenue"]
    .sum()
    .sort_values(ascending=False)
)

print(strategy_revenue)

recommended_strategy
retry_payment                    3999
escalate_to_human                   0
human_review                        0
request_payment_method_update       0
send_recovery_message               0
Name: recovered_revenue, dtype: object


In [ ]:
summary = {
    "total_failed_payments": total_events,
    "total_failed_revenue": int(total_failed_revenue),
    "total_recovered_revenue": int(total_recovered_revenue),
    "recovery_rate_percent": round(recovery_rate, 2),
    "successful_actions": int(successful_actions)
}

print(json.dumps(summary, indent=2))

{
  "total_failed_payments": 5,
  "total_failed_revenue": 14995,
  "total_recovered_revenue": 3999,
  "recovery_rate_percent": 26.67,
  "successful_actions": 5
}


In [ ]:
!pip install -q gradio plotly

In [ ]:
dashboard_metrics = {
    "Failed Payments": int(total_events),
    "Failed Revenue": f"₹{total_failed_revenue:,}",
    "Recovered Revenue": f"₹{total_recovered_revenue:,}",
    "Recovery Rate": f"{recovery_rate:.2f}%",
    "Successful Actions": int(successful_actions)
}

dashboard_metrics

{'Failed Payments': 5,
 'Failed Revenue': '₹14,995',
 'Recovered Revenue': '₹3,999',
 'Recovery Rate': '26.67%',
 'Successful Actions': 5}

In [ ]:
strategy_counts = (
    recovery_logs["recommended_strategy"]
    .value_counts()
    .reset_index()
)

strategy_counts.columns = [
    "Strategy",
    "Count"
]

strategy_counts

,Strategy,Count
0,human_review,1
1,request_payment_method_update,1
2,escalate_to_human,1
3,send_recovery_message,1
4,retry_payment,1


In [ ]:
import gradio as gr
import plotly.express as px
import pandas as pd

# -----------------------------
# Prepare dashboard data
# -----------------------------

display_logs = recovery_logs[
    [
        "transaction_id",
        "customer_name",
        "amount",
        "failure_reason",
        "recommended_strategy",
        "priority",
        "action_success",
        "recovered_revenue"
    ]
].copy()

display_logs["amount"] = display_logs["amount"].apply(
    lambda x: f"₹{int(x):,}"
)

display_logs["recovered_revenue"] = display_logs[
    "recovered_revenue"
].apply(
    lambda x: f"₹{int(x):,}"
)

display_logs["action_success"] = display_logs[
    "action_success"
].apply(
    lambda x: "SUCCESS" if x else "FAILED"
)

# -----------------------------
# Strategy chart
# -----------------------------

fig = px.bar(
    strategy_counts,
    x="Strategy",
    y="Count",
    title="Recovery Strategy Distribution"
)

fig.update_layout(
    xaxis_title="Recovery Strategy",
    yaxis_title="Number of Payments"
)

# -----------------------------
# Dashboard
# -----------------------------

with gr.Blocks(
    title="RecoverAI - Revenue Recovery"
) as dashboard:

    gr.Markdown(
        """
        # 🚀 RecoverAI
        ### AI-Powered Revenue Recovery Agent

        Recover failed and abandoned payments using
        intelligent recovery strategies.
        """
    )

    with gr.Row():

        with gr.Column():
            gr.Markdown(
                f"""
                ## 💰 Failed Revenue

                # ₹{total_failed_revenue:,}
                """
            )

        with gr.Column():
            gr.Markdown(
                f"""
                ## 🔄 Recovered Revenue

                # ₹{total_recovered_revenue:,}
                """
            )

        with gr.Column():
            gr.Markdown(
                f"""
                ## 📈 Recovery Rate

                # {recovery_rate:.2f}%
                """
            )

        with gr.Column():
            gr.Markdown(
                f"""
                ## ⚡ Successful Actions

                # {successful_actions}
                """
            )

    gr.Markdown("---")

    gr.Plot(
        value=fig,
        label="Recovery Strategies"
    )

    gr.Markdown(
        """
        ## 📋 Recovery Events
        """
    )

    gr.Dataframe(
        value=display_logs,
        interactive=False
    )

    gr.Markdown(
        """
        ---
        **RecoverAI**
        Track 3 — AI Revenue Recovery
        """
    )

dashboard.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6c986966c88b83bc2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
# STEP 15.1 — Recovery Priority Score

def calculate_recovery_score(transaction, customer):
    """
    Calculate how important/urgent a failed payment is.

    Higher score = higher recovery priority.
    Score range: 0–100
    """

    score = 0

    amount = float(transaction["amount"])
    attempts = int(transaction["attempt_count"])
    failure_reason = transaction["failure_reason"]

    lifetime_value = float(customer["lifetime_value"])
    successful_payments = int(customer["successful_payments"])
    failed_payments = int(customer["failed_payments"])

    # --------------------------------
    # 1. Payment value
    # --------------------------------

    if amount >= 10000:
        score += 30
    elif amount >= 5000:
        score += 20
    elif amount >= 2000:
        score += 15
    else:
        score += 10

    # --------------------------------
    # 2. Customer lifetime value
    # --------------------------------

    if lifetime_value >= 50000:
        score += 25
    elif lifetime_value >= 20000:
        score += 20
    elif lifetime_value >= 10000:
        score += 15
    else:
        score += 10

    # --------------------------------
    # 3. Failure severity
    # --------------------------------

    if failure_reason == "multiple_failures":
        score += 20

    elif failure_reason == "bank_declined":
        score += 12

    elif failure_reason == "checkout_abandoned":
        score += 10

    elif failure_reason == "authentication_timeout":
        score += 10

    elif failure_reason == "expired_card":
        score += 8

    else:
        score += 15

    # --------------------------------
    # 4. Previous attempts
    # --------------------------------

    if attempts >= 3:
        score += 15
    elif attempts == 2:
        score += 10
    else:
        score += 5

    # --------------------------------
    # 5. Customer reliability
    # --------------------------------

    total_payments = successful_payments + failed_payments

    if total_payments > 0:
        success_rate = successful_payments / total_payments

        if success_rate >= 0.90:
            score += 10
        elif success_rate >= 0.75:
            score += 7
        else:
            score += 3

    # Maximum score = 100
    score = min(score, 100)

    # --------------------------------
    # Priority
    # --------------------------------

    if score >= 70:
        priority = "HIGH"

    elif score >= 40:
        priority = "MEDIUM"

    else:
        priority = "LOW"

    return {
        "recovery_score": score,
        "priority": priority
    }

In [ ]:
# STEP 15.2 — Test Recovery Score

transaction = transactions.iloc[0]

customer = customers[
    customers["customer_id"] == transaction["customer_id"]
].iloc[0]

score_result = calculate_recovery_score(
    transaction,
    customer
)

print(json.dumps(score_result, indent=2))

{
  "recovery_score": 59,
  "priority": "MEDIUM"
}


In [ ]:
# STEP 15.3 — RecoverAI Decision Engine V2

def recoverai_decision_v2(transaction, customer):
    """
    Improved RecoverAI decision engine.

    Combines:
    1. Payment failure analysis
    2. Customer history
    3. Recovery priority score
    4. Recovery strategy selection
    """

    reason = transaction["failure_reason"]
    amount = float(transaction["amount"])
    attempts = int(transaction["attempt_count"])

    successful_payments = int(customer["successful_payments"])
    failed_payments = int(customer["failed_payments"])
    lifetime_value = float(customer["lifetime_value"])

    # --------------------------------
    # Customer success rate
    # --------------------------------

    total_payments = successful_payments + failed_payments

    if total_payments > 0:
        success_rate = round(
            successful_payments / total_payments,
            2
        )
    else:
        success_rate = 0

    # --------------------------------
    # Calculate recovery score
    # --------------------------------

    score_result = calculate_recovery_score(
        transaction,
        customer
    )

    recovery_score = score_result["recovery_score"]

    # --------------------------------
    # Select strategy
    # --------------------------------

    if reason == "multiple_failures":

        strategy = "escalate_to_human"

        decision_reason = (
            "Multiple payment failures indicate that "
            "automatic retries may no longer be appropriate."
        )

        requires_human_approval = True

    elif reason == "expired_card":

        strategy = "request_payment_method_update"

        decision_reason = (
            "The payment method has expired, so the customer "
            "should update their payment details."
        )

        requires_human_approval = False

    elif reason == "checkout_abandoned":

        strategy = "send_recovery_message"

        decision_reason = (
            "The customer abandoned checkout, so a recovery "
            "message can encourage them to complete the payment."
        )

        requires_human_approval = False

    elif reason == "authentication_timeout":

        if attempts <= 1:
            strategy = "retry_payment"
            requires_human_approval = False

            decision_reason = (
                "Authentication timed out once, so a controlled "
                "retry is appropriate."
            )

        else:
            strategy = "escalate_to_human"
            requires_human_approval = True

            decision_reason = (
                "Authentication has already failed multiple times, "
                "so human review is recommended."
            )

    elif reason == "bank_declined":

        if attempts <= 1:
            strategy = "retry_payment"
            requires_human_approval = False

            decision_reason = (
                "The bank declined the payment once and the retry "
                "limit has not been exceeded."
            )

        else:
            strategy = "escalate_to_human"
            requires_human_approval = True

            decision_reason = (
                "The payment has already been retried, so further "
                "automatic attempts should be avoided."
            )

    else:

        strategy = "human_review"
        requires_human_approval = True

        decision_reason = (
            "The failure reason is not recognized by the automated "
            "recovery rules."
        )

    # --------------------------------
    # Adjust priority using score
    # --------------------------------

    if recovery_score >= 70:
        priority = "high"

    elif recovery_score >= 40:
        priority = "medium"

    else:
        priority = "low"

    # --------------------------------
    # Final decision
    # --------------------------------

    return {
        "customer": customer["customer_name"],
        "customer_id": customer["customer_id"],
        "amount": amount,
        "failure_reason": reason,
        "attempt_count": attempts,
        "success_rate": success_rate,
        "lifetime_value": lifetime_value,
        "recovery_score": recovery_score,
        "recommended_strategy": strategy,
        "priority": priority,
        "reason": decision_reason,
        "requires_human_approval": requires_human_approval
    }

In [ ]:
# STEP 15.4 — Test RecoverAI Decision Engine V2

transaction = transactions.iloc[0]

customer = customers[
    customers["customer_id"] == transaction["customer_id"]
].iloc[0]

decision_v2 = recoverai_decision_v2(
    transaction,
    customer
)

print(json.dumps(decision_v2, indent=2))

{
  "customer": "Rahul Sharma",
  "customer_id": "CUS001",
  "amount": 2499.0,
  "failure_reason": "bank_declined",
  "attempt_count": 2,
  "success_rate": 0.88,
  "lifetime_value": 18000.0,
  "recovery_score": 59,
  "recommended_strategy": "escalate_to_human",
  "priority": "medium",
  "reason": "The payment has already been retried, so further automatic attempts should be avoided.",
  "requires_human_approval": true
}


In [29]:
# ============================================================
# RECOVERAI V2 — DECISION ENGINE
# ============================================================

def recoverai_decision_v2(transaction, customer):

    amount = float(transaction["amount"])
    failure_reason = str(transaction["failure_reason"])
    attempts = int(transaction["attempt_count"])

    successful_payments = int(customer["successful_payments"])
    failed_payments = int(customer["failed_payments"])
    lifetime_value = float(customer["lifetime_value"])

    total_payments = successful_payments + failed_payments

    if total_payments > 0:
        success_rate = round(
            successful_payments / total_payments,
            2
        )
    else:
        success_rate = 0

    # Recovery score
    score_result = calculate_recovery_score(
        transaction,
        customer
    )

    recovery_score = score_result["recovery_score"]
    priority = score_result["priority"]

    # Strategy
    if attempts >= 2:

        strategy = "escalate_to_human"

        reason = (
            "The payment has already been retried, "
            "so further automatic attempts should be avoided."
        )

        requires_human_approval = True

    elif failure_reason == "expired_card":

        strategy = "request_payment_method_update"

        reason = (
            "The customer's payment method has expired "
            "and should be updated before another attempt."
        )

        requires_human_approval = False

    elif failure_reason == "multiple_failures":

        strategy = "escalate_to_human"

        reason = (
            "Multiple payment failures indicate that "
            "human review is appropriate."
        )

        requires_human_approval = True

    elif failure_reason == "checkout_abandoned":

        strategy = "send_recovery_message"

        reason = (
            "The customer abandoned checkout, so a "
            "recovery message can encourage completion."
        )

        requires_human_approval = False

    elif failure_reason == "authentication_timeout":

        strategy = "retry_payment"

        reason = (
            "Authentication timed out, so a controlled "
            "retry may recover the payment."
        )

        requires_human_approval = False

    elif failure_reason == "bank_declined":

        strategy = "retry_payment"

        reason = (
            "The bank declined the payment once, so a "
            "controlled retry is appropriate."
        )

        requires_human_approval = False

    else:

        strategy = "human_review"

        reason = (
            "The payment requires additional human review."
        )

        requires_human_approval = True

    return {
        "customer": customer["customer_name"],
        "customer_id": customer["customer_id"],
        "amount": amount,
        "failure_reason": failure_reason,
        "attempt_count": attempts,
        "success_rate": success_rate,
        "lifetime_value": lifetime_value,
        "recovery_score": recovery_score,
        "recommended_strategy": strategy,
        "priority": priority,
        "reason": reason,
        "requires_human_approval": requires_human_approval
    }


print("✅ recoverai_decision_v2 loaded")

✅ recoverai_decision_v2 loaded


In [30]:
print("recoverai_decision_v2" in globals())

True


In [31]:
# STEP 16.1 — RecoverAI Agent V2

def recoverai_agent_v2(transaction_id):
    """
    RecoverAI Agent V2

    Workflow:
    1. Get payment
    2. Get customer
    3. Calculate recovery score
    4. Make intelligent recovery decision
    5. Execute appropriate recovery tool
    6. Return complete result
    """

    # --------------------------------
    # STEP 1 — Get payment
    # --------------------------------

    payment = get_payment_details(transaction_id)

    if not payment["success"]:
        return payment

    # --------------------------------
    # STEP 2 — Get customer
    # --------------------------------

    customer = get_customer_history(
        payment["customer_id"]
    )

    if not customer["success"]:
        return customer

    # --------------------------------
    # STEP 3 — Get dataframe records
    # --------------------------------

    transaction_row = transactions[
        transactions["transaction_id"].astype(str)
        == str(transaction_id)
    ].iloc[0]

    customer_row = customers[
        customers["customer_id"].astype(str)
        == str(payment["customer_id"])
    ].iloc[0]

    # --------------------------------
    # STEP 4 — V2 AI decision
    # --------------------------------

    decision = recoverai_decision_v2(
        transaction_row,
        customer_row
    )

    strategy = decision["recommended_strategy"]

    # --------------------------------
    # STEP 5 — Execute strategy
    # --------------------------------

    if strategy == "retry_payment":

        action_result = retry_payment(
            transaction_id
        )

    elif strategy == "send_recovery_message":

        action_result = send_recovery_message(
            payment["customer_id"],
            payment["amount"]
        )

    elif strategy == "escalate_to_human":

        action_result = escalate_to_human(
            transaction_id,
            decision["reason"]
        )

    elif strategy == "request_payment_method_update":

        action_result = {
            "success": True,
            "action": "request_payment_method_update",
            "customer_id": payment["customer_id"],
            "message": (
                "Customer should update their payment method."
            )
        }

    else:

        action_result = {
            "success": True,
            "action": "human_review",
            "message": (
                "Payment requires human review."
            )
        }

    # --------------------------------
    # STEP 6 — Final agent result
    # --------------------------------

    return {
        "agent": "RecoverAI V2",
        "transaction": payment,
        "customer": customer,
        "decision": decision,
        "action": action_result
    }

In [27]:
print("recoverai_agent_v2 exists:", "recoverai_agent_v2" in globals())

recoverai_agent_v2 exists: True


In [34]:
test_result = recoverai_agent_v2("TXN001")

print(json.dumps(test_result, indent=2))

{
  "agent": "RecoverAI V2",
  "transaction": {
    "success": true,
    "transaction_id": "TXN001",
    "customer_id": "CUS001",
    "amount": 2499,
    "status": "failed",
    "failure_reason": "bank_declined",
    "attempt_count": 2
  },
  "customer": {
    "success": true,
    "customer_id": "CUS001",
    "customer_name": "Rahul Sharma",
    "successful_payments": 7,
    "failed_payments": 1,
    "lifetime_value": 18000
  },
  "decision": {
    "customer": "Rahul Sharma",
    "customer_id": "CUS001",
    "amount": 2499.0,
    "failure_reason": "bank_declined",
    "attempt_count": 2,
    "success_rate": 0.88,
    "lifetime_value": 18000.0,
    "recovery_score": 59,
    "recommended_strategy": "escalate_to_human",
    "priority": "MEDIUM",
    "reason": "The payment has already been retried, so further automatic attempts should be avoided.",
    "requires_human_approval": true
  },
  "action": {
    "success": true,
    "action": "escalate_to_human",
    "transaction_id": "TXN001",


In [ ]:
# STEP 16.3 — Safe Recovery Evaluation
# This does NOT modify transactions or attempt counts.

def evaluate_payment_safely(transaction_id):
    """
    Evaluate a payment without actually modifying
    the transaction dataset.
    """

    payment = get_payment_details(transaction_id)

    if not payment["success"]:
        return payment

    customer = get_customer_history(
        payment["customer_id"]
    )

    if not customer["success"]:
        return customer

    transaction_row = transactions[
        transactions["transaction_id"].astype(str)
        == str(transaction_id)
    ].iloc[0]

    customer_row = customers[
        customers["customer_id"].astype(str)
        == str(payment["customer_id"])
    ].iloc[0]

    # V2 decision
    decision = recoverai_decision_v2(
        transaction_row,
        customer_row
    )

    strategy = decision["recommended_strategy"]

    # --------------------------------
    # Simulate action WITHOUT changing data
    # --------------------------------

    if strategy == "retry_payment":

        action = {
            "success": True,
            "action": "retry_payment",
            "status": "simulated",
            "message": "Payment retry would be attempted."
        }

    elif strategy == "send_recovery_message":

        action = {
            "success": True,
            "action": "send_recovery_message",
            "status": "simulated",
            "message": "Recovery message would be sent."
        }

    elif strategy == "escalate_to_human":

        action = {
            "success": True,
            "action": "escalate_to_human",
            "status": "pending_human_review",
            "message": "Payment requires human review."
        }

    elif strategy == "request_payment_method_update":

        action = {
            "success": True,
            "action": "request_payment_method_update",
            "status": "simulated",
            "message": "Customer would be asked to update payment method."
        }

    else:

        action = {
            "success": True,
            "action": "human_review",
            "status": "pending_human_review",
            "message": "Payment requires human review."
        }

    return {
        "agent": "RecoverAI V2",
        "transaction": payment,
        "customer": customer,
        "decision": decision,
        "action": action
    }

In [ ]:
# STEP 16.4 — Safe Test

safe_result = evaluate_payment_safely("TXN001")

print(json.dumps(safe_result, indent=2))

{
  "agent": "RecoverAI V2",
  "transaction": {
    "success": true,
    "transaction_id": "TXN001",
    "customer_id": "CUS001",
    "amount": 2499,
    "status": "failed",
    "failure_reason": "bank_declined",
    "attempt_count": 2
  },
  "customer": {
    "success": true,
    "customer_id": "CUS001",
    "customer_name": "Rahul Sharma",
    "successful_payments": 7,
    "failed_payments": 1,
    "lifetime_value": 18000
  },
  "decision": {
    "customer": "Rahul Sharma",
    "customer_id": "CUS001",
    "amount": 2499.0,
    "failure_reason": "bank_declined",
    "attempt_count": 2,
    "success_rate": 0.88,
    "lifetime_value": 18000.0,
    "recovery_score": 59,
    "recommended_strategy": "escalate_to_human",
    "priority": "medium",
    "reason": "The payment has already been retried, so further automatic attempts should be avoided.",
    "requires_human_approval": true
  },
  "action": {
    "success": true,
    "action": "escalate_to_human",
    "status": "pending_human_rev

In [ ]:
# STEP 16.5 — Evaluate All Failed/Abandoned Payments

final_agent_results = []

for _, row in failed_transactions.iterrows():

    transaction_id = str(row["transaction_id"])

    try:

        result = evaluate_payment_safely(
            transaction_id
        )

        final_agent_results.append(result)

    except Exception as e:

        print(
            f"Error processing {transaction_id}: {e}"
        )

print(
    "Final payments evaluated:",
    len(final_agent_results)
)

Final payments evaluated: 5


In [ ]:
# STEP 16.6 — Final RecoverAI Results Table

final_records = []

for result in final_agent_results:

    transaction = result["transaction"]
    customer = result["customer"]
    decision = result["decision"]
    action = result["action"]

    final_records.append({

        "transaction_id":
            transaction["transaction_id"],

        "customer_name":
            customer["customer_name"],

        "amount":
            transaction["amount"],

        "failure_reason":
            transaction["failure_reason"],

        "attempt_count":
            transaction["attempt_count"],

        "success_rate":
            decision["success_rate"],

        "lifetime_value":
            decision["lifetime_value"],

        "recovery_score":
            decision["recovery_score"],

        "priority":
            decision["priority"],

        "recommended_strategy":
            decision["recommended_strategy"],

        "human_approval":
            decision["requires_human_approval"],

        "action":
            action["action"],

        "action_status":
            action["status"]
    })

final_results_df = pd.DataFrame(final_records)

final_results_df

,transaction_id,customer_name,amount,failure_reason,attempt_count,success_rate,lifetime_value,recovery_score,priority,recommended_strategy,human_approval,action,action_status
0,TXN001,Rahul Sharma,2499,bank_declined,2,0.88,18000.0,59,medium,escalate_to_human,True,escalate_to_human,pending_human_review
1,TXN002,Aisha Khan,999,expired_card,1,0.80,7500.0,40,medium,request_payment_method_update,False,request_payment_method_update,simulated
2,TXN003,Vikram Singh,5999,multiple_failures,3,0.80,42000.0,82,high,escalate_to_human,True,escalate_to_human,pending_human_review
3,TXN004,Sara Patel,1499,checkout_abandoned,0,1.00,12500.0,50,medium,send_recovery_message,False,send_recovery_message,simulated
4,TXN005,Arjun Mehta,3999,authentication_timeout,2,0.90,29000.0,65,medium,escalate_to_human,True,escalate_to_human,pending_human_review


In [ ]:
!pip install -q streamlit plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 80.4 MB/s eta 0:00:00


In [ ]:
final_results_df

,transaction_id,customer_name,amount,failure_reason,attempt_count,success_rate,lifetime_value,recovery_score,priority,recommended_strategy,human_approval,action,action_status
0,TXN001,Rahul Sharma,2499,bank_declined,2,0.88,18000.0,59,medium,escalate_to_human,True,escalate_to_human,pending_human_review
1,TXN002,Aisha Khan,999,expired_card,1,0.80,7500.0,40,medium,request_payment_method_update,False,request_payment_method_update,simulated
2,TXN003,Vikram Singh,5999,multiple_failures,3,0.80,42000.0,82,high,escalate_to_human,True,escalate_to_human,pending_human_review
3,TXN004,Sara Patel,1499,checkout_abandoned,0,1.00,12500.0,50,medium,send_recovery_message,False,send_recovery_message,simulated
4,TXN005,Arjun Mehta,3999,authentication_timeout,2,0.90,29000.0,65,medium,escalate_to_human,True,escalate_to_human,pending_human_review


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="RecoverAI",
    page_icon="🚀",
    layout="wide"
)

# --------------------------------------------------
# Demo data
# --------------------------------------------------

data = [
    {
        "transaction_id": "TXN001",
        "customer_name": "Rahul Sharma",
        "amount": 2499,
        "failure_reason": "bank_declined",
        "attempt_count": 2,
        "success_rate": 0.88,
        "lifetime_value": 18000,
        "recovery_score": 59,
        "priority": "medium",
        "recommended_strategy": "escalate_to_human",
        "human_approval": True,
        "action": "escalate_to_human",
        "action_status": "pending_human_review"
    }
]

df = pd.DataFrame(data)

# --------------------------------------------------
# Header
# --------------------------------------------------

st.title("🚀 RecoverAI")

st.subheader(
    "AI-Powered Revenue Recovery Agent"
)

st.write(
    """
    RecoverAI analyzes failed and abandoned payments,
    evaluates customer history, calculates recovery
    priority, and recommends a safe recovery strategy.
    """
)

st.divider()

# --------------------------------------------------
# Metrics
# --------------------------------------------------

total_failed_payments = len(df)

total_failed_revenue = df["amount"].sum()

high_priority_cases = len(
    df[df["priority"] == "high"]
)

human_review_cases = len(
    df[df["human_approval"] == True]
)

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "Failed Payments",
        total_failed_payments
    )

with col2:
    st.metric(
        "Failed Revenue",
        f"₹{total_failed_revenue:,}"
    )

with col3:
    st.metric(
        "High Priority",
        high_priority_cases
    )

with col4:
    st.metric(
        "Human Review",
        human_review_cases
    )

st.divider()

# --------------------------------------------------
# Strategy chart
# --------------------------------------------------

st.subheader(
    "📊 Recovery Strategy Distribution"
)

strategy_counts = (
    df["recommended_strategy"]
    .value_counts()
    .reset_index()
)

strategy_counts.columns = [
    "Strategy",
    "Count"
]

fig = px.bar(
    strategy_counts,
    x="Strategy",
    y="Count",
    title="Recommended Recovery Strategies"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

# --------------------------------------------------
# Recovery cases
# --------------------------------------------------

st.subheader(
    "🤖 AI Recovery Decisions"
)

st.dataframe(
    df[
        [
            "transaction_id",
            "customer_name",
            "amount",
            "failure_reason",
            "recovery_score",
            "priority",
            "recommended_strategy",
            "human_approval"
        ]
    ],
    use_container_width=True,
    hide_index=True
)

# --------------------------------------------------
# Human review queue
# --------------------------------------------------

st.subheader(
    "👨‍💼 Human Approval Queue"
)

human_queue = df[
    df["human_approval"] == True
]

if len(human_queue) > 0:

    st.warning(
        f"{len(human_queue)} payment(s) require human review."
    )

    st.dataframe(
        human_queue[
            [
                "transaction_id",
                "customer_name",
                "amount",
                "failure_reason",
                "recovery_score",
                "recommended_strategy"
            ]
        ],
        use_container_width=True,
        hide_index=True
    )

else:

    st.success(
        "No payments currently require human approval."
    )

# --------------------------------------------------
# Footer
# --------------------------------------------------

st.divider()

st.caption(
    "RecoverAI — Track 3: AI Revenue Recovery"
)

Writing app.py


In [ ]:
!streamlit run app.py &>/content/streamlit.log &

!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://cyan-wasps-greet.loca.lt
^C


In [8]:
print("transactions exists:", "transactions" in globals())
print("customers exists:", "customers" in globals())

print(transactions.shape)
print(customers.shape)

transactions exists: True
customers exists: True
(5, 7)
(5, 5)


In [37]:
# STEP 17 — Save RecoverAI data files

import os

os.makedirs("/content/recoverai/data", exist_ok=True)

transactions.to_csv(
    "/content/recoverai/data/transactions.csv",
    index=False
)

customers.to_csv(
    "/content/recoverai/data/customers.csv",
    index=False
)

print("Data files created successfully!")
print()
print("transactions.csv:", len(transactions), "rows")
print("customers.csv:", len(customers), "rows")

Data files created successfully!

transactions.csv: 5 rows
customers.csv: 5 rows


In [44]:
# STEP 18 — Save recovery logs

recovery_logs.to_csv(
    "/content/recoverai/data/recovery_logs.csv",
    index=False
)

print("Recovery logs saved successfully!")
print("recovery_logs.csv:", len(recovery_logs), "rows")

Recovery logs saved successfully!
recovery_logs.csv: 5 rows


In [45]:
# STEP 19 — Verify RecoverAI data files

import os

data_path = "/content/recoverai/data"

print("Checking RecoverAI data files...")
print()

for filename in [
    "transactions.csv",
    "customers.csv",
    "recovery_logs.csv"
]:
    filepath = os.path.join(data_path, filename)

    print(
        f"{filename}:",
        "✅ EXISTS" if os.path.exists(filepath) else "❌ MISSING"
    )

print()
print("Files in data folder:")
print(os.listdir(data_path))

Checking RecoverAI data files...

transactions.csv: ✅ EXISTS
customers.csv: ✅ EXISTS
recovery_logs.csv: ✅ EXISTS

Files in data folder:
['customers.csv', 'transactions.csv', 'recovery_logs.csv']


In [46]:
# STEP 20 — Prepare Streamlit project

import os

project_path = "/content/recoverai"

folders = [
    project_path,
    f"{project_path}/data",
    f"{project_path}/.streamlit"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Streamlit project structure ready")
print()
print("Project:", project_path)
print("Folders:")
print(os.listdir(project_path))

✅ Streamlit project structure ready

Project: /content/recoverai
Folders:
['.streamlit', 'data']


In [49]:
# STEP 21 — Check Streamlit app

app_path = "/content/recoverai/app.py"

if os.path.exists(app_path):
    print("✅ app.py exists")
    print("Size:", os.path.getsize(app_path), "bytes")
else:
    print("❌ app.py is missing")

✅ app.py exists
Size: 3837 bytes


In [48]:
# STEP 22 — Create RecoverAI Streamlit app

app_code = r'''
import streamlit as st
import pandas as pd
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------

DATA_DIR = "/content/recoverai/data"

TRANSACTIONS_FILE = os.path.join(DATA_DIR, "transactions.csv")
CUSTOMERS_FILE = os.path.join(DATA_DIR, "customers.csv")
RECOVERY_LOGS_FILE = os.path.join(DATA_DIR, "recovery_logs.csv")


# --------------------------------------------------
# Load data
# --------------------------------------------------

transactions = pd.read_csv(TRANSACTIONS_FILE)
customers = pd.read_csv(CUSTOMERS_FILE)

if os.path.exists(RECOVERY_LOGS_FILE):
    recovery_logs = pd.read_csv(RECOVERY_LOGS_FILE)
else:
    recovery_logs = pd.DataFrame()


# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="RecoverAI",
    page_icon="💳",
    layout="wide"
)


# --------------------------------------------------
# Header
# --------------------------------------------------

st.title("💳 RecoverAI")
st.subheader("AI Revenue Recovery Dashboard")

st.markdown(
    "Recover failed and abandoned payments using "
    "intelligent recovery strategies."
)


# --------------------------------------------------
# Metrics
# --------------------------------------------------

total_transactions = len(transactions)

failed_count = len(
    transactions[
        transactions["status"].isin(["failed", "abandoned"])
    ]
)

recovered_count = 0

if not recovery_logs.empty and "success" in recovery_logs.columns:
    recovered_count = int(
        recovery_logs["success"].astype(bool).sum()
    )

total_revenue = 0

if "amount" in transactions.columns:
    total_revenue = transactions["amount"].sum()


col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "Total Transactions",
        total_transactions
    )

with col2:
    st.metric(
        "Failed / Abandoned",
        failed_count
    )

with col3:
    st.metric(
        "Recovery Events",
        len(recovery_logs)
    )

with col4:
    st.metric(
        "Transaction Value",
        f"${total_revenue:,.2f}"
    )


# --------------------------------------------------
# Transactions
# --------------------------------------------------

st.divider()

st.header("📋 Transactions")

st.dataframe(
    transactions,
    use_container_width=True,
    hide_index=True
)


# --------------------------------------------------
# Recovery logs
# --------------------------------------------------

st.divider()

st.header("🔄 Recovery Activity")

if recovery_logs.empty:
    st.info("No recovery events available.")
else:
    st.dataframe(
        recovery_logs,
        use_container_width=True,
        hide_index=True
    )


# --------------------------------------------------
# Human review queue
# --------------------------------------------------

st.divider()

st.header("👤 Human Review Queue")

if not recovery_logs.empty and "recommended_strategy" in recovery_logs.columns:

    human_queue = recovery_logs[
        recovery_logs["recommended_strategy"].isin(
            ["escalate_to_human", "human_review"]
        )
    ]

    if len(human_queue) > 0:
        st.warning(
            f"{len(human_queue)} payment(s) require human review."
        )

        st.dataframe(
            human_queue,
            use_container_width=True,
            hide_index=True
        )
    else:
        st.success(
            "No payments currently require human approval."
        )

else:
    st.info(
        "Human review information is not available yet."
    )


# --------------------------------------------------
# Footer
# --------------------------------------------------

st.divider()

st.caption(
    "RecoverAI — AI Revenue Recovery"
)
'''

with open("/content/recoverai/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully!")
print("Path: /content/recoverai/app.py")
print("Size:", os.path.getsize("/content/recoverai/app.py"), "bytes")

✅ app.py created successfully!
Path: /content/recoverai/app.py
Size: 3837 bytes


In [52]:
# STEP 24A — Install Streamlit

!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.8 MB/s eta 0:00:00


In [53]:
# STEP 24B — Verify Streamlit installation

import shutil

streamlit_path = shutil.which("streamlit")

if streamlit_path:
    print("✅ Streamlit installed")
    print("Path:", streamlit_path)
else:
    print("❌ Streamlit is still not installed")

✅ Streamlit installed
Path: /usr/local/bin/streamlit


In [54]:
# STEP 24C — Start RecoverAI Streamlit

import subprocess
import time

# Stop old Streamlit processes
subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True
)

time.sleep(2)

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/recoverai/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("✅ Streamlit started")
print("Process ID:", streamlit_process.pid)

✅ Streamlit started
Process ID: 25706


In [55]:
# STEP 25 — Check RecoverAI Streamlit

import requests

try:
    response = requests.get(
        "http://localhost:8501",
        timeout=10
    )

    print("✅ Streamlit is running!")
    print("Status code:", response.status_code)

except Exception as e:
    print("❌ Streamlit is not responding")
    print("Error:", e)

print("\n--- Streamlit log ---")

with open("/content/streamlit.log", "r") as f:
    print(f.read()[-3000:])

✅ Streamlit is running!
Status code: 200

--- Streamlit log ---


2026-08-22 07:49:46.283 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.178.223.125:8501




In [56]:
# STEP 26 — Prepare RecoverAI for GitHub

import os

project_path = "/content/recoverai"

os.makedirs(f"{project_path}/data", exist_ok=True)

files_to_check = [
    f"{project_path}/app.py",
    f"{project_path}/data/transactions.csv",
    f"{project_path}/data/customers.csv",
    f"{project_path}/data/recovery_logs.csv",
]

print("RecoverAI GitHub deployment files:")
print()

for file_path in files_to_check:
    if os.path.exists(file_path):
        print("✅", file_path)
    else:
        print("❌ MISSING:", file_path)

RecoverAI GitHub deployment files:

✅ /content/recoverai/app.py
✅ /content/recoverai/data/transactions.csv
✅ /content/recoverai/data/customers.csv
✅ /content/recoverai/data/recovery_logs.csv
